---
title: "Temporal resampling/aggregation"
# categories: [xarray]
# date: 2025-04-11
---

In [1]:
import xarray as xr

from util import generate_3d_dataset

## Some random data

In [2]:
lat, lon, time = 40, 60, 120
ds = generate_3d_dataset(lat, lon, time)
ds

<xarray.Dataset> Size: 2MB
Dimensions:  (lat: 40, lon: 60, time: 120)
Coordinates:
  * lat      (lat) int64 320B 0 1 2 3 4 5 6 7 8 9 ... 31 32 33 34 35 36 37 38 39
  * lon      (lon) int64 480B 0 1 2 3 4 5 6 7 8 9 ... 51 52 53 54 55 56 57 58 59
  * time     (time) datetime64[ns] 960B 2021-01-01 2021-01-02 ... 2021-04-30
Data variables:
    test     (lat, lon, time) float64 2MB dask.array<chunksize=(4, 6, 120), meta=np.ndarray>

## Resample to monthly interval

In [3]:
da_monthly_mean = ds.test.resample(time="1MS").mean()

In [4]:
def custom_agg_func(da: xr.DataArray) -> xr.DataArray:
    # dummy operation - could by anything
    return da.sum(dim="time") / da.count(dim="time")

In [5]:
da_monthly_mean_custom = ds.test.resample(time="1MS").apply(custom_agg_func)

In [6]:
assert (da_monthly_mean_custom == da_monthly_mean).all()

## Upsampling and gap-filling

I'll reuse the monthly aggregated data set and the timesteps of the original data set to illustrate reindexing and gap-filling, although this has nothing to do with the above workflow.

In [7]:
# reindex with some time coordinates
da_reindexed = da_monthly_mean.reindex(time=ds.time)

/home/andi/miniconda3/envs/satpy/lib/python3.13/site-packages/dask/array/core.py:4988: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(
/home/andi/miniconda3/envs/satpy/lib/python3.13/site-packages/dask/array/core.py:4988: PerformanceWarning: Increasing number of chunks by factor of 100
  result = blockwise(


In [8]:
# nearest neighbour or some other interpolation scheme
ds_filled = da_reindexed.chunk(dict(time=-1)).interpolate_na(
    dim="time", method="nearest", fill_value="extrapolate"
)

In [9]:
da_monthly_mean.isel(lat=0, lon=0).load()

<xarray.DataArray 'test' (time: 4)> Size: 32B
array([0.43229829, 0.52233213, 0.56107932, 0.43978903])
Coordinates:
    lat      int64 8B 0
    lon      int64 8B 0
  * time     (time) datetime64[ns] 32B 2021-01-01 2021-02-01 ... 2021-04-01

In [10]:
da_reindexed.isel(lat=0, lon=0).load()

<xarray.DataArray 'test' (time: 120)> Size: 960B
array([0.43229829,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan, 0.52233213,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan, 0.56107932,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
       0.43978903,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan,
              nan,        nan,        nan,        nan,        nan])
Coordinates:
    lat      int64 8B 0
    lon      int64 8B 0
  * time     (time) datetime64[ns] 960B 2021-01-01 2021-01-02 ... 2021-04-30

In [11]:
ds_filled.isel(lat=0, lon=0).load()

<xarray.DataArray 'test' (time: 120)> Size: 960B
array([0.43229829, 0.43229829, 0.43229829, 0.43229829, 0.43229829,
       0.43229829, 0.43229829, 0.43229829, 0.43229829, 0.43229829,
       0.43229829, 0.43229829, 0.43229829, 0.43229829, 0.43229829,
       0.43229829, 0.52233213, 0.52233213, 0.52233213, 0.52233213,
       0.52233213, 0.52233213, 0.52233213, 0.52233213, 0.52233213,
       0.52233213, 0.52233213, 0.52233213, 0.52233213, 0.52233213,
       0.52233213, 0.52233213, 0.52233213, 0.52233213, 0.52233213,
       0.52233213, 0.52233213, 0.52233213, 0.52233213, 0.52233213,
       0.52233213, 0.52233213, 0.52233213, 0.52233213, 0.52233213,
       0.52233213, 0.56107932, 0.56107932, 0.56107932, 0.56107932,
       0.56107932, 0.56107932, 0.56107932, 0.56107932, 0.56107932,
       0.56107932, 0.56107932, 0.56107932, 0.56107932, 0.56107932,
       0.56107932, 0.56107932, 0.56107932, 0.56107932, 0.56107932,
       0.56107932, 0.56107932, 0.56107932, 0.56107932, 0.56107932,
       0.56107932, 0.56107932, 0.56107932, 0.56107932, 0.56107932,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903,
       0.43978903, 0.43978903, 0.43978903, 0.43978903, 0.43978903])
Coordinates:
    lat      int64 8B 0
    lon      int64 8B 0
  * time     (time) datetime64[ns] 960B 2021-01-01 2021-01-02 ... 2021-04-30